In [5]:
# !pip install nltk

In [6]:
import pandas as pd
import numpy as np
import re
from nltk.stem import SnowballStemmer
from nltk.tokenize import word_tokenize

In [7]:
df = pd.read_csv("/content/app_reviews_labeled.csv")

In [8]:
df = df[:35000]

In [9]:
df.shape

(35000, 4)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35000 entries, 0 to 34999
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   content        34993 non-null  object
 1   score          35000 non-null  int64 
 2   thumbsUpCount  35000 non-null  int64 
 3   label          35000 non-null  object
dtypes: int64(2), object(2)
memory usage: 1.1+ MB


In [11]:
df.isnull().sum()

,0
content,7
score,0
thumbsUpCount,0
label,0


In [12]:
df.dropna(inplace=True)

In [13]:
df.drop('score', axis=1, inplace=True)

In [14]:
df.sample(6)

,content,thumbsUpCount,label
23891,we yw see awards dress f2 sweet sewer was Wwww...,0,negative
12874,Good,3,positive
28492,IT'S MY GO-TO KEYBOARD APPLICATION!... I have ...,6,positive
33006,Ok,0,positive
627,my contacts are not crossing over,1,neutral
25742,The app was fairly good till the latest update...,54,negative


In [15]:
# Convert the sentiment into the numbers
def sentiment_into_number(sentiment):
  if sentiment == 'negative':
    return 0
  elif sentiment == 'neutral':
    return 1
  elif sentiment == 'positive':
    return 2

df['label'] = df['label'].apply(sentiment_into_number)

In [16]:
df.head()

,content,thumbsUpCount,label
0,Working with this app is so difficult. Default...,566,0
1,I would give it 0 stars if possible. No option...,189,0
2,Dear Google.. I found a very critical bug..cus...,105,0
3,Worst interface ever......can't even add a new...,403,0
4,"While opening a saved contact entry, this app ...",277,1


In [17]:
import nltk
nltk.download('punkt')
import re
import unicodedata
from nltk.stem import PorterStemmer

def lower_text(text):
  """
  This function is use to lower the playstore reviews content/text
  """
  return text.lower()


def clean_text(text):
    text = str(text)

    # HTML
    text = re.sub(r"<.*?>", "", text)

    # URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Emojis / symbols
    text = "".join(
        char
        for char in text
        if not unicodedata.category(char).startswith("So")
    )

    # Extra whitespace
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def apply_stemming(text):
    """Apply Snowball stemming to a sentence."""
    stemmer = PorterStemmer()
    words = word_tokenize(text)
    stemmed_words = [stemmer.stem(word) for word in words]
    return " ".join(stemmed_words)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [18]:
import nltk
nltk.download('punkt_tab', quiet=True)


def text_preprocessing(text):
  text_lower = lower_text(text)
  cleaned_content = clean_text(text_lower)
  # stemmed_text = apply_stemming(cleaned_content)

  return cleaned_content

df['content'] = df['content'].apply(text_preprocessing)

In [19]:
from sklearn.model_selection import train_test_split

X = df["content"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [20]:
X_train.shape

(27994,)

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [22]:
from sklearn.metrics import classification_report
from sklearn.svm import LinearSVC

# Initialize model with your specific parameters
model = LinearSVC(
    C=1.1709879033298216,
    tol=0.000008137560594377145,
    loss='hinge',
    fit_intercept=True,
    class_weight='balanced',
    max_iter=4324
)

# Train
model.fit(X_train_tfidf, y_train)

# Predict
y_pred = model.predict(X_test_tfidf)

# Classification report
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.84      0.90      0.87      3092
           1       0.75      0.70      0.72      1518
           2       0.90      0.85      0.87      2389

    accuracy                           0.84      6999
   macro avg       0.83      0.82      0.82      6999
weighted avg       0.84      0.84      0.84      6999



In [23]:
sample = pd.DataFrame({
    "content": [
        "Amazing app!",
        "Too many bugs and crashes every day.",
        "It is okay, nothing special.",
        "Beautiful app",
        "Absolutely love this! Best user experience ever.",
        "The recent update completely broke the login screen.",
        "It does what it says, but the UI could be better.",
        "Highly recommended! Saves me so much time daily.",
        "Total waste of time. It freezes constantly on my phone.",
        "Just downloaded it. It works fine for now.",
        "Incredibly fast and very intuitive to navigate.",
        "Extremely disappointed. Terrible customer support.",
        "An average application, standard features like others.",
        "Perfect tool! I cannot imagine my routine without it."
    ]
})

# 4. Preprocess text into a separate column (FIXED 'df' error and preserved original text)
sample['content_clean'] = sample['content'].apply(text_preprocessing)

# 5. Transform using the existing fitted vectorizer
content_tfidf = tfidf_vectorizer.transform(sample['content_clean'])

# 6. Predict and append labels (FIXED model reference)
sample['label'] = model.predict(content_tfidf)

# 7. Display results side-by-side
print("\nPredicted Sample Sentiments:")


Predicted Sample Sentiments:


In [24]:
sample[['content', 'label']]

,content,label
0,Amazing app!,2
1,Too many bugs and crashes every day.,0
2,"It is okay, nothing special.",1
3,Beautiful app,2
4,Absolutely love this! Best user experience ever.,2
5,The recent update completely broke the login s...,0
6,"It does what it says, but the UI could be better.",0
7,Highly recommended! Saves me so much time daily.,2
8,Total waste of time. It freezes constantly on ...,0
9,Just downloaded it. It works fine for now.,2


In [30]:
# 2. Define the mapping dictionary
class_map = {0: 'negative', 1: 'neutral', 2: 'positive'}

# 3. Create DataFrame preserving the original X_test index
df_results = pd.DataFrame({
    'Text_Content': X_test,
    'Predicted_Label': y_pred
})

# 4. Map the numeric labels to text sentiments
df_results['Predicted_Label'] = df_results['Predicted_Label'].map(class_map)

In [31]:
df_results['Predicted_Label'].value_counts()

,count
Predicted_Label,
negative,3290
positive,2277
neutral,1432


In [27]:
# !pip install sentence-transformers umap-learn hdbscan keybert -q

In [28]:
# import pandas as pd
# import numpy as np
# import warnings
# warnings.filterwarnings("ignore")

# from sentence_transformers import SentenceTransformer
# import umap
# import hdbscan
# from keybert import KeyBERT

# # ============================================================
# # STEP 1: Load models once (reused across all sentiment groups)
# # ============================================================
# print("Loading embedding + keyword models...")
# embedder = SentenceTransformer("all-MiniLM-L6-v2")
# kw_model = KeyBERT(embedder)  # reuse same model, avoids loading twice

# # ============================================================
# # STEP 2: Embed ALL reviews once (efficient — don't redo per group)
# # ============================================================
# print("Embedding all reviews...")
# all_embeddings = embedder.encode(
#     df_results["Text_Content"].astype(str).tolist(),
#     batch_size=64,
#     show_progress_bar=True
# )
# df_results["_embedding_idx"] = range(len(df_results))  # track row -> embedding row

# # ============================================================
# # STEP 3: Function to cluster + extract keywords for ONE sentiment
# # ============================================================
# def cluster_sentiment_group(df_results, embeddings, sentiment_label, top_n_keywords=3):
#     subset = df_results[df_results["Predicted_Label"] == sentiment_label]
#     n = len(subset)

#     if n < 30:
#         print(f"[{sentiment_label}] Too few reviews ({n}) to cluster meaningfully. Skipping.")
#         return None, {}

#     sub_embeddings = embeddings[subset["_embedding_idx"].values]

#     # ---- UMAP dimensionality reduction ----
#     reducer = umap.UMAP(
#         n_components=min(10, n - 2),   # guard against tiny groups
#         n_neighbors=min(15, n - 1),
#         min_dist=0.0,
#         metric="cosine",
#         random_state=42
#     )
#     reduced = reducer.fit_transform(sub_embeddings)

#     # ---- Adaptive min_cluster_size: ~2% of group, bounded [15, 80] ----
#     min_cluster_size = int(np.clip(n * 0.02, 15, 80))

#     clusterer = hdbscan.HDBSCAN(
#         min_cluster_size=min_cluster_size,
#         min_samples=max(5, min_cluster_size // 5),
#         metric="euclidean",
#         cluster_selection_method="eom"
#     )
#     cluster_labels = clusterer.fit_predict(reduced)

#     subset = subset.copy()
#     subset["cluster"] = cluster_labels

#     n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
#     n_noise = (cluster_labels == -1).sum()
#     print(f"\n[{sentiment_label.upper()}] {n} reviews -> {n_clusters} clusters, "
#           f"{n_noise} noise ({n_noise/n*100:.1f}%)")

#     # ---- KeyBERT keyword extraction per cluster ----
#     cluster_names = {}
#     for cluster_id in sorted(set(cluster_labels)):
#         if cluster_id == -1:
#             cluster_names[-1] = "Uncategorized"
#             continue

#         cluster_reviews = subset[subset["cluster"] == cluster_id]["Text_Content"].astype(str).tolist()
#         combined_text = " ".join(cluster_reviews[:200])[:20000]  # cap length for speed

#         try:
#             keywords = kw_model.extract_keywords(
#                 combined_text,
#                 keyphrase_ngram_range=(1, 2),
#                 stop_words="english",
#                 top_n=top_n_keywords,
#                 use_mmr=True,
#                 diversity=0.5
#             )
#             topic_name = " | ".join([kw[0] for kw in keywords]) if keywords else "N/A"
#         except Exception as e:
#             topic_name = "N/A"

#         cluster_names[cluster_id] = topic_name
#         print(f"  Cluster {cluster_id:2d} ({len(cluster_reviews):4d} reviews): {topic_name}")

#     subset["topic"] = subset["cluster"].map(cluster_names)
#     return subset, cluster_names

# # ============================================================
# # STEP 4: Run for each sentiment
# # ============================================================
# results_by_sentiment = {}
# topics_by_sentiment = {}

# for sentiment in ["negative", "neutral", "positive"]:
#     clustered_df, topics = cluster_sentiment_group(df_results, all_embeddings, sentiment)
#     results_by_sentiment[sentiment] = clustered_df
#     topics_by_sentiment[sentiment] = topics

# # ============================================================
# # STEP 5: Combine everything back into one dataframe (optional)
# # ============================================================
# df_final = pd.concat([v for v in results_by_sentiment.values() if v is not None], ignore_index=True)

In [29]:
# app_summary = {}

# for sentiment, topics in topics_by_sentiment.items():
#     keyword_list = []
#     for cluster_id, name in topics.items():
#         if cluster_id == -1:
#             continue
#         keyword_list.extend(name.split(" | "))
#     app_summary[sentiment] = keyword_list

# import json
# print(json.dumps(app_summary, indent=2))